# CSE598 Capstone Runnable Baseline

This notebook runs the same dependency-free course-policy Q&A baseline as `run_baseline.py`. Run every cell from top to bottom. If the repository files are not present, the first code cell clones the public repository automatically.

## Requirements

- Python 3.10 or newer
- No third-party runtime packages
- No API keys or environment variables

In [1]:
import json
import os
import subprocess
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'run_baseline.py').exists():
    clone_dir = project_root / 'cse598-capstone-baseline'
    if not clone_dir.exists():
        subprocess.run([
            'git', 'clone',
            'https://github.com/Deep-nayak007/cse598-capstone-baseline.git',
            str(clone_dir),
        ], check=True)
    os.chdir(clone_dir)
    project_root = Path.cwd()

from run_baseline import answer_question, build_vectors

kb_path = project_root / 'examples/course_faq.json'
input_path = project_root / 'examples/test_questions.json'
output_path = project_root / 'output/baseline_output.json'

assert kb_path.exists(), 'The knowledge base is missing.'
assert input_path.exists(), 'The sample input file is missing.'
print('Project files located successfully.')

Project files located successfully.


In [2]:
records = json.loads(kb_path.read_text())
questions = json.loads(input_path.read_text())
print(f'Loaded {len(records)} policy entries and {len(questions)} test questions.')
questions

Loaded 5 policy entries and 3 test questions.


['When do I need to submit the capstone proposal?',
 'What files should my README mention?',
 'Can the baseline be simple?']

In [3]:
vectors, idf = build_vectors(records)
results = [answer_question(question, records, vectors, idf) for question in questions]

for result in results:
    print(f"[{result['status']}] {result['question']}")
    print(f"source={result['matched_source']} confidence={result['confidence']}")
    print(result['answer'])
    print()

[answered] When do I need to submit the capstone proposal?
source=capstone_due_date confidence=0.463
The capstone project proposal must be submitted to Canvas by September 6 at 11:59 PM Phoenix Time.

[answered] What files should my README mention?
source=readme_requirements confidence=0.509
The README should explain dependencies, setup steps, required API keys or environment variables, exact commands or notebook cells to run, and where to find input and output files.

[answered] Can the baseline be simple?
source=baseline_requirement confidence=0.315
The baseline does not need to be strong, but it must be runnable, reproducible, and testable on at least one concrete example.



In [4]:
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(results, indent=2) + '\n')
print(f'Wrote {len(results)} answers to {output_path.relative_to(project_root)}')

Wrote 3 answers to output/baseline_output.json


In [5]:
expected_sources = ['capstone_due_date', 'readme_requirements', 'baseline_requirement']
actual_sources = [result['matched_source'] for result in results]
assert actual_sources == expected_sources

unrelated = answer_question(
    'How do I bake sourdough bread?', records, vectors, idf
)
assert unrelated['status'] == 'needs_clarification'
print('Notebook checks passed: expected matches and fallback behavior verified.')

Notebook checks passed: expected matches and fallback behavior verified.
